In [ ]:
import numpy as np
import pandas as pd
import string
import re
import time

#**Project 2: Analyzing the Similarity of Essays with Natural Language Processing**

The dataset that was used for this project can be found on https://www.kaggle.com/competitions/learning-agency-lab-automated-essay-scoring-2/data


To access my conversation with ChatGPT that influenced this report, the link can be found here:
https://chatgpt.com/c/67117041-c3f8-8000-815c-43a7fc147e6f

##1.) Load dataset

In [ ]:
from google.colab import files
x = files.upload()

Saving train.csv to train.csv


In [ ]:
essay_df = pd.read_csv('train.csv')

##2.) Visualize dataset and confirm its operation

In [ ]:
essay_df.head()

,essay_id,full_text,score
0,000d118,Many people have car where they live. The thin...,3
1,000fe60,I am a scientist at NASA that is discussing th...,3
2,001ab80,People always wish they had the same technolog...,4
3,001bdc0,"We all heard about Venus, the planet without a...",4
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3


In [ ]:
essay_df.tail()

,essay_id,full_text,score
17302,ffd378d,"the story "" The Challenge of Exploing Venus "" ...",2
17303,ffddf1f,Technology has changed a lot of ways that we l...,4
17304,fff016d,If you don't like sitting around all day than ...,2
17305,fffb49b,"In ""The Challenge of Exporing Venus,"" the auth...",1
17306,fffed3e,Venus is worthy place to study but dangerous. ...,2


In [ ]:
# 17,307 essays will be analyzed for similarity.
essay_df.shape

(17307, 3)

##3.) Clean and preprocess the data

**Remove all punctuation and uppercase to improve the algorithms' accuracy**

In [ ]:
def clean_essay(text):
  text = text.replace('PROPER_NAME', '')
  text = text.lower()                                 # Convert csv file to all lowercase.
  text = re.sub(r'[^\w\s]', '', text)                 # Remove all punctuation.

  return text

essay_df['cleaned_essay'] = essay_df['full_text'].apply(clean_essay)

**Remove unncessary columns from dataset**

In [ ]:
# Removes the third column "score".
# The column represents the quality of the essay writing and will be of no use to analyzing text similarity.

essay_df.drop(columns=['score', 'full_text'], inplace=True)

In [ ]:
# "full_text" and "score" columns are removed.
# "cleaned_essay" column added.
essay_df.head()

,essay_id,cleaned_essay
0,000d118,many people have car where they live the thing...
1,000fe60,i am a scientist at nasa that is discussing th...
2,001ab80,people always wish they had the same technolog...
3,001bdc0,we all heard about venus the planet without al...
4,002ba53,dear state senator\n\nthis is a letter to argu...


##4.) Apply NLP algorithms

###**Method 1: Word Frequency-Based Approach (Count Vectorizer)**

####**Run the algorithm**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix


count_vectorizer_start_time = time.time()
similar_essay_count = 5

# Create the CountVectorizer.
vectorizer = CountVectorizer(stop_words='english')
count_vectorizer_matrix = vectorizer.fit_transform(essay_df['cleaned_essay'])

count_vectorizer_df = pd.DataFrame(count_vectorizer_matrix.toarray(), columns=vectorizer.get_feature_names_out())

# Convert to sparse matrix to prevent using all available RAM in Google Colab.
sparse_matrix = csr_matrix(count_vectorizer_df)

# Normalize with cosine similarity.
cosine_similarity_matrix = cosine_similarity(sparse_matrix)

count_vectorizer_end_time = time.time()
count_vectorizer_elapsed = count_vectorizer_end_time - count_vectorizer_start_time

####**Determine the most similar essay pairs in each matrix row**

In [ ]:
essay_analysis_start_time = time.time()


# Stores highest similarity score from each matrix row.
most_similar_essay_in_row = np.zeros([essay_df.shape[0], 3])
least_similar_essay_in_row = np.zeros([essay_df.shape[0], 3])

for i in range(cosine_similarity_matrix.shape[0]):
  # Gathers a specific row of scores from the cosine similarity matrix.
  similarity_scores = cosine_similarity_matrix[i]

  # Gathers the indices of each similiarity score, then sorts the row by highest value first.
  sorted_indices = np.argsort(similarity_scores)[ : :-1]

  # Find the most similar essay (i.e., one with the highest cosine similarity) in each row in the matrix.
  for j in sorted_indices:
    if i != j:
      most_similar_indices = j
      break

  most_similar_essay_in_row[i][0] = i                                                       # Matrix row.
  most_similar_essay_in_row[i][1] = most_similar_indices                                    # Matrix column (index 'j').
  most_similar_essay_in_row[i][2] = similarity_scores[most_similar_indices]                 # Cosine similarity score.

   # Gathers the indices of each similiarity score, then sorts the row by lowest value first.
  sorted_indices = np.argsort(similarity_scores)

  # Find the least similar essay (i.e., one with the lowest cosine similarity) in each row in the matrix.
  for j in sorted_indices:
    if i != j:
      least_similar_indices = j
      break

  least_similar_essay_in_row[i][0] = i                                                       # Matrix row.
  least_similar_essay_in_row[i][1] = most_similar_indices                                    # Matrix column.
  least_similar_essay_in_row[i][2] = similarity_scores[least_similar_indices]                # Cosine similarity score.

essay_analysis_end_time = time.time()
essay_analysis_elapsed = essay_analysis_end_time - essay_analysis_start_time

####**Sort and print top 5 essay pairs with highest (and lowest) similarity**

In [ ]:
sort_and_print_start_time = time.time()

previous_score = 0
index = 0                             # Increments every single 'while' iteration.
iteration = 0                         # Increments ONLY when score is unique to avoid repeated pairs (i.e, Essays 2 and 5 == Essays 5 and 2).


# Places the essay pairs with the highest cosine similarity scores first (descending order).
top_essay_rows = most_similar_essay_in_row[np.argsort(most_similar_essay_in_row[ : , 2])][ : :-1]

# Stores top 5 essay pairs in list to extract their indices and essay strings.
essay_1 = []
essay_2 = []

print("5 Most Similar Essay Pairs")
while (iteration != similar_essay_count):
  if top_essay_rows[index][2] == previous_score:
    index += 1
    continue

  previous_score = top_essay_rows[index][2]
  print(f"\t{iteration + 1}.) Essay {top_essay_rows[index][0]:.0f} is similar to Essay {top_essay_rows[index][1]:.0f} (score: {top_essay_rows[index][2]:.5f}).")

  essay_1.append(top_essay_rows[index][0])
  essay_2.append(top_essay_rows[index][1])

  index += 1
  iteration += 1
  if iteration == similar_essay_count:
    break

print("\n")

previous_score = 0
index = 0
iteration = 0


# Places the essay pairs with the lowest cosine similarity scores first (ascending order).
bottom_essay_rows = least_similar_essay_in_row[np.argsort(least_similar_essay_in_row[ : , 2])]

print("5 Least Similar Essay Pairs")
while (iteration != similar_essay_count):
  if bottom_essay_rows[index][2] == previous_score:
    index += 1
    continue

  previous_score = bottom_essay_rows[index][2]
  print(f"\t{iteration + 1}.) Essay {bottom_essay_rows[index][0]:.0f} is similar to Essay {bottom_essay_rows[index][1]:.0f} (score: {bottom_essay_rows[index][2]:.5f}).")

  index += 1
  iteration += 1
  if iteration == similar_essay_count:
    break

sort_and_print_end_time = time.time()
sort_and_print_elapsed = sort_and_print_end_time - sort_and_print_start_time
count_v_total_end_time = count_vectorizer_elapsed + essay_analysis_elapsed + sort_and_print_elapsed


print(f"\nCount Vectorizer Execution Time: {count_vectorizer_elapsed:.3f} seconds")
print(f"Total Execution Time: {count_v_total_end_time:.3f} seconds")

5 Most Similar Essay Pairs
	1.) Essay 793 is similar to Essay 15844 (score: 0.99801).
	2.) Essay 14394 is similar to Essay 10969 (score: 0.99676).
	3.) Essay 9014 is similar to Essay 9252 (score: 0.99645).
	4.) Essay 6423 is similar to Essay 11131 (score: 0.99153).
	5.) Essay 2789 is similar to Essay 7284 (score: 0.90399).


5 Least Similar Essay Pairs
	1.) Essay 1085 is similar to Essay 16175 (score: 0.00082).
	2.) Essay 1757 is similar to Essay 13556 (score: 0.00085).
	3.) Essay 14684 is similar to Essay 3277 (score: 0.00096).
	4.) Essay 4727 is similar to Essay 6029 (score: 0.00100).
	5.) Essay 12937 is similar to Essay 14045 (score: 0.00100).

Count Vectorizer Execution Time: 43.245 seconds
Total Execution Time: 101.101 seconds


####**Visualize similarities between the top 5 essay pairs**

In [ ]:
essay_1_keys = []
essay_1_essays = []
essay_2_keys = []
essay_2_essays = []

results_df = pd.DataFrame(columns=['Essay 1 Index', 'Essay 1 Key', 'Essay 2 Index', 'Essay 2 Key', 'Essay 1 Sample', 'Essay 2 Sample'])

for i in range(5):
  essay_1_keys.append(essay_df['essay_id'][essay_1[i]])
  essay_1_essays.append(essay_df['cleaned_essay'][essay_1[i]])

  essay_2_keys.append(essay_df['essay_id'][essay_2[i]])
  essay_2_essays.append(essay_df['cleaned_essay'][essay_2[i]])

results_df['Essay 1 Index'] = [int(value) for value in essay_1]
results_df['Essay 1 Key'] = essay_1_keys
results_df['Essay 2 Index'] = [int(value) for value in essay_2]
results_df['Essay 2 Key'] = essay_2_keys

results_df['Essay 1 Sample'] = essay_1_essays
results_df['Essay 2 Sample'] = essay_2_essays

results_df.head()

,Essay 1 Index,Essay 1 Key,Essay 2 Index,Essay 2 Key,Essay 1 Sample,Essay 2 Sample
0,793,0c8039c,15844,e9be80d,i think people should participate in the seago...,i think people should participate in the seago...
1,14394,d43da53,10969,a1cde0f,in this essay i will talk about why you should...,in this essay i will talk about why you should...
2,9014,85215b2,9252,88a671e,hi i am and i really recomend that you join t...,hi i am luke and i really recomend that you jo...
3,6423,6017fea,11131,a423c92,i really dont like this new computer thing at ...,i really dont like this new computer thing at ...
4,2789,29aa983,7284,6d25307,a new hom\n\nwhould you send someone to explor...,benefits of researching a new planet\n\nwhould...


###**Method 2: TF-IDF Similarity**


####**Run the algorithm**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix


tfidf_start_time = time.time()
similar_essay_count = 5

# Create the TF-IDF vectorier.
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_vectorizer_matrix = vectorizer.fit_transform(essay_df['cleaned_essay'])

tfidf_vectorizer_df = pd.DataFrame(tfidf_vectorizer_matrix.toarray(), columns=vectorizer.get_feature_names_out())

# Convert to sparse matrix to prevent using all available RAM in Google Colab.
sparse_matrix = csr_matrix(tfidf_vectorizer_df)

# Normalize with cosine similarity.
cosine_similarity_matrix = cosine_similarity(sparse_matrix)

tfidf_end_time = time.time()
tfidf_elapsed = tfidf_end_time - tfidf_start_time

####**Determine the most similar essay pairs in each matrix row**

In [ ]:
essay_analysis_start_time = time.time()


# Stores highest similarity score from each matrix row.
most_similar_essay_in_row = np.zeros([essay_df.shape[0], 3])
least_similar_essay_in_row = np.zeros([essay_df.shape[0], 3])

for i in range(cosine_similarity_matrix.shape[0]):
  # Gathers a specific row of scores from the cosine similarity matrix.
  similarity_scores = cosine_similarity_matrix[i]

  # Gathers the indices of each similiarity score, then sorts the row by highest value first.
  sorted_indices = np.argsort(similarity_scores)[ : :-1]

  # Find the most similar essay (i.e., one with the highest cosine similarity) in each row in the matrix.
  for j in sorted_indices:
    if i != j:
      most_similar_indices = j
      break

  most_similar_essay_in_row[i][0] = i                                                       # Matrix row.
  most_similar_essay_in_row[i][1] = most_similar_indices                                    # Matrix column.
  most_similar_essay_in_row[i][2] = similarity_scores[most_similar_indices]                 # Cosine similarity score.

   # Gathers the indices of each similiarity score, then sorts the row by lowest value first.
  sorted_indices = np.argsort(similarity_scores)

  # Find the least similar essay (i.e., one with the lowest cosine similarity) in each row in the matrix.
  for j in sorted_indices:
    if i != j:
      least_similar_indices = j
      break

  least_similar_essay_in_row[i][0] = i                                                       # Matrix row.
  least_similar_essay_in_row[i][1] = most_similar_indices                                    # Matrix column.
  least_similar_essay_in_row[i][2] = similarity_scores[least_similar_indices]                # Cosine similarity score.

essay_analysis_end_time = time.time()
essay_analysis_elapsed = essay_analysis_end_time - essay_analysis_start_time

####**Sort and print top 5 essay pairs with highest (and lowest) similarity**

In [ ]:
sort_and_print_start_time = time.time()

previous_score = 0
index = 0                             # Increments every single 'while' iteration.
iteration = 0                         # Increments ONLY when score is unique to avoid repeated pairs (i.e, Essays 2 and 5 == Essays 5 and 2).


# Places the essay pairs with the highest cosine similarity scores first (descending order).
top_essay_rows = most_similar_essay_in_row[np.argsort(most_similar_essay_in_row[ : , 2])][ : :-1]

# Stores top 5 essay pairs in list to extract their indices and essay strings.
essay_1 = []
essay_2 = []

print("5 Most Similar Essay Pairs")
while (iteration != similar_essay_count):
  if top_essay_rows[index][2] == previous_score:
    index += 1
    continue

  previous_score = top_essay_rows[index][2]
  print(f"\t{iteration + 1}.) Essay {top_essay_rows[index][0]:.0f} is similar to Essay {top_essay_rows[index][1]:.0f} (score: {top_essay_rows[index][2]:.5f}).")

  essay_1.append(top_essay_rows[index][0])
  essay_2.append(top_essay_rows[index][1])

  index += 1
  iteration += 1
  if iteration == similar_essay_count:
    break

print("\n")

previous_score = 0
index = 0
iteration = 0


# Places the essay pairs with the lowest cosine similarity scores first (ascending order).
bottom_essay_rows = least_similar_essay_in_row[np.argsort(least_similar_essay_in_row[ : , 2])]

print("5 Least Similar Essay Pairs")
while (iteration != similar_essay_count):
  if bottom_essay_rows[index][2] == previous_score:
    index += 1
    continue

  previous_score = bottom_essay_rows[index][2]
  print(f"\t{iteration + 1}.) Essay {bottom_essay_rows[index][0]:.0f} is similar to Essay {bottom_essay_rows[index][1]:.0f} (score: {bottom_essay_rows[index][2]:.5f}).")

  index += 1
  iteration += 1
  if iteration == similar_essay_count:
    break


sort_and_print_end_time = time.time()
sort_and_print_elapsed = sort_and_print_end_time - sort_and_print_start_time
tfidf_total_end_time = tfidf_elapsed + essay_analysis_elapsed + sort_and_print_elapsed


print(f"\nTF-IDF Execution Time: {tfidf_elapsed:.3f} seconds")
print(f"Total Execution Time: {tfidf_total_end_time:.3f} seconds")

5 Most Similar Essay Pairs
	1.) Essay 15844 is similar to Essay 793 (score: 0.99816).
	2.) Essay 9014 is similar to Essay 9252 (score: 0.99442).
	3.) Essay 14394 is similar to Essay 10969 (score: 0.98392).
	4.) Essay 11131 is similar to Essay 6423 (score: 0.91546).
	5.) Essay 8406 is similar to Essay 8973 (score: 0.78142).


5 Least Similar Essay Pairs
	1.) Essay 2919 is similar to Essay 4560 (score: 0.00021).
	2.) Essay 2698 is similar to Essay 6992 (score: 0.00022).
	3.) Essay 11555 is similar to Essay 16960 (score: 0.00022).
	4.) Essay 12548 is similar to Essay 6992 (score: 0.00023).
	5.) Essay 13173 is similar to Essay 15858 (score: 0.00024).

TF-IDF Execution Time: 53.640 seconds
Total Execution Time: 112.772 seconds


####**Visualize similarities between the top 5 essay pairs**

In [ ]:
essay_1_keys = []
essay_1_essays = []
essay_2_keys = []
essay_2_essays = []

results_df = pd.DataFrame(columns=['Essay 1 Index', 'Essay 1 Key', 'Essay 2 Index', 'Essay 2 Key', 'Essay 1 Sample', 'Essay 2 Sample'])

for i in range(5):
  essay_1_keys.append(essay_df['essay_id'][essay_1[i]])
  essay_1_essays.append(essay_df['cleaned_essay'][essay_1[i]])

  essay_2_keys.append(essay_df['essay_id'][essay_2[i]])
  essay_2_essays.append(essay_df['cleaned_essay'][essay_2[i]])

results_df['Essay 1 Index'] = [int(value) for value in essay_1]
results_df['Essay 1 Key'] = essay_1_keys
results_df['Essay 2 Index'] = [int(value) for value in essay_2]
results_df['Essay 2 Key'] = essay_2_keys

results_df['Essay 1 Sample'] = essay_1_essays
results_df['Essay 2 Sample'] = essay_2_essays

results_df.head()

,Essay 1 Index,Essay 1 Key,Essay 2 Index,Essay 2 Key,Essay 1 Sample,Essay 2 Sample
0,15844,e9be80d,793,0c8039c,i think people should participate in the seago...,i think people should participate in the seago...
1,9014,85215b2,9252,88a671e,hi i am and i really recomend that you join t...,hi i am luke and i really recomend that you jo...
2,14394,d43da53,10969,a1cde0f,in this essay i will talk about why you should...,in this essay i will talk about why you should...
3,11131,a423c92,6423,6017fea,i really dont like this new computer thing at ...,i really dont like this new computer thing at ...
4,8406,7cdf8b2,8973,84a1b1a,does electoral college work\n\ntoday i am goin...,dear state senator\n\ndo you think that we sho...


###**Method 3: Hashing Vectorizer**

####**Run the algorithm**

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix


hashing_vectorizer_start_time = time.time()
similar_essay_count = 5

# Create the Hashing Vectorizer.
vectorizer = HashingVectorizer(n_features=20, norm=None, alternate_sign=False)
hashing_vectorizer_matrix = vectorizer.fit_transform(essay_df['cleaned_essay'])

hashing_vectorizer_df = pd.DataFrame(hashing_vectorizer_matrix.toarray())

# Convert to sparse matrix to prevent using all available RAM in Google Colab.
sparse_matrix = csr_matrix(hashing_vectorizer_df)

# Normalize with cosine similarity.
cosine_similarity_matrix = cosine_similarity(sparse_matrix)

hashing_vectorizer_end_time = time.time()
hashing_vectorizer_elapsed = hashing_vectorizer_end_time - hashing_vectorizer_start_time

####**Determine the most similar essay pairs in each matrix row**

In [ ]:
essay_analysis_start_time = time.time()


# Stores highest similarity score from each matrix row.
most_similar_essay_in_row = np.zeros([essay_df.shape[0], 3])
least_similar_essay_in_row = np.zeros([essay_df.shape[0], 3])


for i in range(cosine_similarity_matrix.shape[0]):
  # Gathers a specific row of scores from the cosine similarity matrix.
  similarity_scores = cosine_similarity_matrix[i]

  # Gathers the indices of each similiarity score, then sorts the row by highest value first.
  sorted_indices = np.argsort(similarity_scores)[ : :-1]

  # Find the most similar essay (i.e., one with the highest cosine similarity) in each row in the matrix.
  for j in sorted_indices:
    if i != j:
      most_similar_indices = j
      break

  most_similar_essay_in_row[i][0] = i                                                       # Matrix row.
  most_similar_essay_in_row[i][1] = most_similar_indices                                    # Matrix column.
  most_similar_essay_in_row[i][2] = similarity_scores[most_similar_indices]                 # Cosine similarity score.

   # Gathers the indices of each similiarity score, then sorts the row by lowest value first.
  sorted_indices = np.argsort(similarity_scores)

  # Find the least similar essay (i.e., one with the lowest cosine similarity) in each row in the matrix.
  for j in sorted_indices:
    if i != j:
      least_similar_indices = j
      break

  least_similar_essay_in_row[i][0] = i                                                       # Matrix row.
  least_similar_essay_in_row[i][1] = most_similar_indices                                    # Matrix column.
  least_similar_essay_in_row[i][2] = similarity_scores[least_similar_indices]                # Cosine similarity score.

essay_analysis_end_time = time.time()
essay_analysis_elapsed = essay_analysis_end_time - essay_analysis_start_time

####**Sort and print top 5 essay pairs with highest (and lowest) similarity**

In [ ]:
sort_and_print_start_time = time.time()

previous_score = 0
index = 0                             # Increments every single 'while' iteration.
iteration = 0                         # Increments ONLY when score is unique to avoid repeated pairs (i.e, Essays 2 and 5 == Essays 5 and 2).


# Places the essay pairs with the highest cosine similarity scores first (descending order).
top_essay_rows = most_similar_essay_in_row[np.argsort(most_similar_essay_in_row[ : , 2])][ : :-1]

# Stores top 5 essay pairs in list to extract their indices and essay strings.
essay_1 = []
essay_2 = []

print("5 Most Similar Essay Pairs")
while (iteration != similar_essay_count):
  if top_essay_rows[index][2] == previous_score:
    index += 1
    continue

  previous_score = top_essay_rows[index][2]
  print(f"\t{iteration + 1}.) Essay {top_essay_rows[index][0]:.0f} is similar to Essay {top_essay_rows[index][1]:.0f} (score: {top_essay_rows[index][2]:.5f}).")

  essay_1.append(top_essay_rows[index][0])
  essay_2.append(top_essay_rows[index][1])

  index += 1
  iteration += 1
  if iteration == similar_essay_count:
    break

print("\n")

previous_score = 0
index = 0
iteration = 0


# Places the essay pairs with the lowest cosine similarity scores first (ascending order).
bottom_essay_rows = least_similar_essay_in_row[np.argsort(least_similar_essay_in_row[ : , 2])]

print("5 Least Similar Essay Pairs")
while (iteration != similar_essay_count):
  if bottom_essay_rows[index][2] == previous_score:
    index += 1
    continue

  previous_score = bottom_essay_rows[index][2]
  print(f"\t{iteration + 1}.) Essay {bottom_essay_rows[index][0]:.0f} is similar to Essay {bottom_essay_rows[index][1]:.0f} (score: {bottom_essay_rows[index][2]:.5f}).")

  index += 1
  iteration += 1
  if iteration == similar_essay_count:
    break


sort_and_print_end_time = time.time()
sort_and_print_elapsed = sort_and_print_end_time - sort_and_print_start_time
hash_v_total_end_time = hashing_vectorizer_elapsed + essay_analysis_elapsed + sort_and_print_elapsed


print(f"\nHashing Vectorizer Execution Time: {hashing_vectorizer_elapsed:.3f} seconds")
print(f"Total Execution Time: {hash_v_total_end_time:.3f} seconds")

5 Most Similar Essay Pairs
	1.) Essay 793 is similar to Essay 15844 (score: 0.99993).
	2.) Essay 9252 is similar to Essay 9014 (score: 0.99981).
	3.) Essay 10969 is similar to Essay 14394 (score: 0.99975).
	4.) Essay 11131 is similar to Essay 6423 (score: 0.99951).
	5.) Essay 16819 is similar to Essay 6592 (score: 0.99587).


5 Least Similar Essay Pairs
	1.) Essay 12736 is similar to Essay 10571 (score: 0.44768).
	2.) Essay 11841 is similar to Essay 6389 (score: 0.45081).
	3.) Essay 2383 is similar to Essay 16492 (score: 0.45276).
	4.) Essay 2043 is similar to Essay 10114 (score: 0.46121).
	5.) Essay 10933 is similar to Essay 14670 (score: 0.46448).

Hashing Vectorizer Execution Time: 36.306 seconds
Total Execution Time: 93.147 seconds


####**Visualize similarities between the top 5 essay pairs**

In [ ]:
essay_1_keys = []
essay_1_essays = []
essay_2_keys = []
essay_2_essays = []

results_df = pd.DataFrame(columns=['Essay 1 Index', 'Essay 1 Key', 'Essay 2 Index', 'Essay 2 Key', 'Essay 1 Sample', 'Essay 2 Sample'])

for i in range(5):
  essay_1_keys.append(essay_df['essay_id'][essay_1[i]])
  essay_1_essays.append(essay_df['cleaned_essay'][essay_1[i]])

  essay_2_keys.append(essay_df['essay_id'][essay_2[i]])
  essay_2_essays.append(essay_df['cleaned_essay'][essay_2[i]])

results_df['Essay 1 Index'] = [int(value) for value in essay_1]
results_df['Essay 1 Key'] = essay_1_keys
results_df['Essay 2 Index'] = [int(value) for value in essay_2]
results_df['Essay 2 Key'] = essay_2_keys

results_df['Essay 1 Sample'] = essay_1_essays
results_df['Essay 2 Sample'] = essay_2_essays

results_df.head()

,Essay 1 Index,Essay 1 Key,Essay 2 Index,Essay 2 Key,Essay 1 Sample,Essay 2 Sample
0,793,0c8039c,15844,e9be80d,i think people should participate in the seago...,i think people should participate in the seago...
1,9252,88a671e,9014,85215b2,hi i am luke and i really recomend that you jo...,hi i am and i really recomend that you join t...
2,10969,a1cde0f,14394,d43da53,in this essay i will talk about why you should...,in this essay i will talk about why you should...
3,11131,a423c92,6423,6017fea,i really dont like this new computer thing at ...,i really dont like this new computer thing at ...
4,16819,f834b3c,6592,6265245,the effect of cars in our world today has grow...,car use all over the world has tried to be red...


##5.) Find random essay pairs with high/low similarity scores

Any randomized essay pairs' similarity scores will be compared to the similarity percentage calculated on https://toolsaday.com/text-analysis/similarity-checker.

The code below will execute for all three NLP algorithms.


In [ ]:
import random

# Two random Kaggle dataset rows are selected.
essay_A = random.randint(0, essay_df.shape[0] - 1)
essay_B = random.randint(0, essay_df.shape[0] - 1)

# The while-loop definition's numerical values are adjusted to find essay pairs with high similarity or low similarity.
while cosine_similarity_matrix[essay_A][essay_B] > 0.72 or cosine_similarity_matrix[essay_A][essay_B] > 0.99999:
  essay_A = random.randint(0, essay_df.shape[0] - 1)
  essay_B = random.randint(0, essay_df.shape[0] - 1)

  if cosine_similarity_matrix[essay_A][essay_B] <= 0.72 and essay_A != essay_B:
    break


# Essay 1
print("Essay 1:")
print("\tNumber:", essay_A)
print("\tKey: ", essay_df['essay_id'][essay_A])
print("\n ", essay_df['cleaned_essay'][essay_A])

# Essay 2
print("\n\nEssay 2:")
print("\tNumber:", essay_B)
print("\tKey: ", essay_df['essay_id'][essay_B])
print("\n ", essay_df['cleaned_essay'][essay_B])


# Cosine similarity score for pair
print("\nSimilarity score:", cosine_similarity_matrix[essay_A][essay_B])

Essay 1:
	Number: 6597
	Key:  62729e1

  should we study venus or throw away the idea because all the dangers venus is the closest planet to earth therefore if anything happened to earth we would have to move to our twin planet even though venus has the hottest surface temperature out of all the planets and we humans wouldnt last a second on the planet scientist are finding ways to study the planet and see if we possibly could create life there

many researchers are working on innovations that would allow our machines to last long enough to contribute meaningfully to our knowledge of venus the author includes ways like this to prove how and why we should get knowledge about venus he includes in paragraph 5 about nasa and their ideas for sending humans to study the planet nasas possible solution to the hostile conditions on the surface of venus would allow scientists to float above the fray

there are many reasons to why we souldnt study venus and get close to the planet but there are a